In [1]:
# Write images from a folder into a GIF or MP4.
# GIF_START/GIF_END are list indices after sorting, using Python's [start:end] convention.
from pathlib import Path
import json
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np

GIF_IMAGES_DIR = "./viz_vehicle/Generalization_PedestriansOnRoad_1085_route0_07_09_17_41_28_hipad_f2d/images"
GIF_START = 30
GIF_END = 110
GIF_FPS = 10
# TODO
mp4_mode = False
OUTPUT_PATH = Path("./gifs") / f"hipad_brake_new.{'mp4' if mp4_mode else 'gif'}"
ACTION_LOG_PATH = Path(GIF_IMAGES_DIR).parent / "activation_actions.jsonl"

def frame_idx_from_path(path):
    try:
        return int(Path(path).stem)
    except ValueError:
        return None

def image_sort_key(path):
    frame_idx = frame_idx_from_path(path) if "frame_idx_from_path" in globals() else None
    return (0, frame_idx) if frame_idx is not None else (1, path.name)


def list_images(image_dir):
    image_dir = Path(image_dir)
    image_paths = []
    for suffix in ("*.png", "*.jpg", "*.jpeg"):
        image_paths.extend(image_dir.glob(suffix))
    return sorted(image_paths, key=image_sort_key)


def load_activation_rows(action_log_path):
    action_log_path = Path(action_log_path)
    if not action_log_path.exists():
        return {}

    rows = {}
    with action_log_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows[int(row["frame"])] = row
    return rows


def activation_is_on(activation_alpha):
    if activation_alpha is None:
        return False
    if isinstance(activation_alpha, (list, tuple)):
        return any(abs(float(value)) > 1e-9 for value in activation_alpha)
    return abs(float(activation_alpha)) > 1e-9


def load_overlay_font(frame_height):
    font_size = max(16, frame_height // 28)
    try:
        return ImageFont.truetype("DejaVuSans-Bold.ttf", font_size)
    except OSError:
        return ImageFont.load_default()


def draw_activation_label(frame, activation_on):
    frame = frame.convert("RGBA")
    overlay = Image.new("RGBA", frame.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(overlay)
    font = load_overlay_font(frame.height)
    text = f"activation: {'on' if activation_on else 'off'}"
    x, y = 10, 10
    padding = 6
    left, top, right, bottom = draw.textbbox((x, y), text, font=font)
    draw.rounded_rectangle(
        (left - padding, top - padding, right + padding, bottom + padding),
        radius=4,
        fill=(0, 0, 0, 160),
    )
    draw.text((x, y), text, fill=(255, 255, 255, 255), font=font)
    return Image.alpha_composite(frame, overlay).convert("RGB")


image_paths = list_images(GIF_IMAGES_DIR)
selected_paths = image_paths[GIF_START:GIF_END]
if not selected_paths:
    raise FileNotFoundError(
        f"No images found in slice [{GIF_START}:{GIF_END}] under {Path(GIF_IMAGES_DIR).resolve()}"
    )

activation_rows = load_activation_rows(ACTION_LOG_PATH)

frames = []
for path in selected_paths:
    with Image.open(path) as image:
        width, height = image.size
        frame = image.convert("RGB").resize((width // 2, height // 2))
    frame_idx = frame_idx_from_path(path)
    row = activation_rows.get(frame_idx, {})
    frames.append(draw_activation_label(frame, activation_is_on(row.get("activation_alpha"))))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
if mp4_mode:
    width, height = frames[0].size
    writer = cv2.VideoWriter(
        str(OUTPUT_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        GIF_FPS,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not open MP4 writer for {OUTPUT_PATH}")
    for frame in frames:
        writer.write(cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR))
    writer.release()
else:
    gif_frames = [frame.convert("P", palette=Image.ADAPTIVE) for frame in frames]
    gif_frames[0].save(
        OUTPUT_PATH,
        save_all=True,
        append_images=gif_frames[1:],
        duration=round(1000 / GIF_FPS),
        loop=0,
        optimize=False,
    )

print(f"{'mp4' if mp4_mode else 'gif'} frames: {len(frames)} from slice [{GIF_START}:{GIF_END}]")
print("saved:", OUTPUT_PATH.resolve())

gif frames: 80 from slice [30:110]
saved: /media/user/data1/shu_wei/fail2drive/gifs/hipad_brake_new.gif


In [ ]:
import json, math

alphas = ['0', '02', '04', '06', '08', '10']
for alpha in alphas:
    path = f"results/alpha{alpha}/1085_peds.json"

    with open(path) as f:
        data = json.load(f)

    record = data["_checkpoint"]["records"][0]
    scores = record["scores"]
    infractions = record["infractions"]

    ds = float(scores["score_composed"])
    rc = float(scores["score_route"])
    ip = float(scores["score_penalty"])

    ignored = {"min_speed_infractions", "outside_route_lanes"}
    success = all(
        not entries
        for name, entries in infractions.items()
        if name not in ignored
    )
    sr = 100.0 if success else 0.0

    hm = 0.0 if ds == 0 or sr == 0 else 2.0 / ((1.0 / ds) + (1.0 / sr))

    print(f"Alphas: {alpha}")
    # print(f"Status: {record['status']}")
    print(f"DS: {ds:.3f}")
    # print(f"RC: {rc:.3f}")
    # print(f"Infraction penalty: {ip:.6f}")
    print(f"SR: {sr:.1f}")
    print(f"HM: {hm:.3f}")

In [5]:
import json, math

alpha = '05'
path = f"results/alpha{alpha}/1085_peds.json"

with open(path) as f:
    data = json.load(f)

record = data["_checkpoint"]["records"][0]
scores = record["scores"]
infractions = record["infractions"]

ds = float(scores["score_composed"])
rc = float(scores["score_route"])
ip = float(scores["score_penalty"])

ignored = {"min_speed_infractions", "outside_route_lanes"}
success = all(
    not entries
    for name, entries in infractions.items()
    if name not in ignored
)
sr = 100.0 if success else 0.0

hm = 0.0 if ds == 0 or sr == 0 else 2.0 / ((1.0 / ds) + (1.0 / sr))

print(f"Alphas: {alpha}")
# print(f"Status: {record['status']}")
print(f"DS: {ds:.3f}")
# print(f"RC: {rc:.3f}")
# print(f"Infraction penalty: {ip:.6f}")
print(f"SR: {sr:.1f}")
print(f"HM: {hm:.3f}")

Alphas: 05
DS: 10.223
SR: 0.0
HM: 0.000
